> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Application Cost Intelligence & Unit Economics**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../04_application_cost_intelligence.ipynb`.

# Notebook 04: Application Cost Intelligence & Unit Economics

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 04 of 12  
**Focus:** Fully-Burdened Application TCO, Unit Economics Modeling, Pareto Spend Concentration, Elasticity Dynamics  

---

## 1. Business Problem

Most enterprise IT organizations track application software licenses, but severely underestimate true **Application Total Cost of Ownership (TCO)**. Software licenses often represent less than 35% of an application's true operational burden:
- **Internal & External Labor:** Dedicated engineering, operations, and support staff.
- **Dedicated & Shared Infrastructure:** Cloud compute instances, storage buckets, database clusters, and physical servers.
- **Third-Party Support & Vendor Contracts:** Ancillary monitoring, tooling, and maintenance.

Furthermore, aggregate cost alone does not tell an executive whether an application is efficient. An application costing ₹15 Crores supporting 10,000 active users is vastly more cost-effective than an application costing ₹5 Crores supporting only 50 users. We must evaluate **Unit Economics**.

---

## 2. Core Concept & Formulation

### 2.1 Fully-Burdened Application TCO
$$\text{TCO}(\text{App}) = \text{Software} + \text{Internal Labor} + \text{Cloud} + \text{Infrastructure} + \text{Vendor Services}$$

### 2.2 Unit Economics Metrics
To normalize cost by operational delivery, we define:
$$\text{Annual Cost per Active User} = \frac{\text{Annual TCO}}{\text{Average Monthly Active Users}}$$
$$\text{Cost per Digital Transaction} = \frac{\text{Annual TCO}}{\text{Annual Transaction Volume}}$$
*Safe Zero Handling:* In cases where an application operates in batch or back-office mode with zero interactive users, the denominator is guarded to prevent division-by-zero errors.

### 2.3 Pareto Concentration (80/20 Rule)
We compute cumulative spend share to identify the critical subset of applications that drive the majority of enterprise technology costs.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config
from tvi.cost_analytics import calculate_application_tco, get_application_profile
from tvi.database import get_database

config = load_config()
db = get_database(config.database.path)

# Compute fully burdened TCO and unit economics
app_tco_df = calculate_application_tco()
print("Application Cost Intelligence Table Compiled:")
display(app_tco_df[["application_id", "application_name", "lifecycle_status", "annual_tco", "avg_monthly_users", "annual_cost_per_user", "cost_per_transaction", "cumulative_spend_pct"]].head(8))

Application Cost Intelligence Table Compiled:


---

## 3. Cost Stack Decomposition Across Categories

We unpack the composition of each application's annual TCO across **all cost categories**: Software, Internal Labor, Cloud, and Infrastructure:

In [2]:
# Top 15 Applications Cost Stack Breakdown
top_15 = app_tco_df.head(15)

plt.figure(figsize=(11, 5.5))
bar_w = 0.65
x = np.arange(len(top_15))

p1 = plt.bar(x, top_15["software_cost"] / 1e7, width=bar_w, label="Software / License", color="#1f77b4")
p2 = plt.bar(x, top_15["people_cost"] / 1e7, bottom=top_15["software_cost"] / 1e7, width=bar_w, label="People / Labor", color="#ff7f0e")
p3 = plt.bar(x, top_15["cloud_cost"] / 1e7, bottom=(top_15["software_cost"] + top_15["people_cost"]) / 1e7, width=bar_w, label="Cloud Services", color="#2ca02c")
p4 = plt.bar(x, top_15["infra_cost"] / 1e7, bottom=(top_15["software_cost"] + top_15["people_cost"] + top_15["cloud_cost"]) / 1e7, width=bar_w, label="Shared Infrastructure", color="#d62728")

plt.xticks(x, top_15["application_id"], rotation=45, ha="right", fontsize=9)
plt.ylabel("Annual Spend (₹ Crores)", fontsize=11)
plt.title("Top 15 Applications: Fully-Burdened Cost Stack Decomposition", fontsize=13)
plt.legend(loc="upper right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---

## 4. Pareto Spend Concentration & Lorenz Curve

We plot the Lorenz curve of cumulative portfolio spend against cumulative application count to identify portfolio concentration:

In [3]:
# Pareto 80/20 Lorenz Curve Analysis
sorted_spend = app_tco_df["annual_tco"].sort_values(ascending=False).values
cum_spend = np.cumsum(sorted_spend) / np.sum(sorted_spend) * 100.0
app_ranks = np.arange(1, len(sorted_spend) + 1) / len(sorted_spend) * 100.0

plt.figure(figsize=(8, 4.5))
plt.plot(app_ranks, cum_spend, color="#2b5c8f", linewidth=3, label="Cumulative Spend % (Actual)")
plt.plot([0, 100], [0, 100], color="grey", linestyle="--", label="Equal Distribution Baseline")
plt.axhline(80, color="#d62728", linestyle=":", alpha=0.8, label="80% Spend Threshold")

# Find app count reaching 80% spend
idx_80 = np.where(cum_spend >= 80.0)[0][0] + 1
pct_apps_80 = (idx_80 / len(sorted_spend)) * 100.0
plt.axvline(pct_apps_80, color="#d62728", linestyle=":")
plt.scatter([pct_apps_80], [80], color="#d62728", zorder=5)
plt.annotate(f"{idx_80} apps ({pct_apps_80:.1f}%) = 80% Spend", xy=(pct_apps_80, 80),
             xytext=(pct_apps_80 + 5, 70), arrowprops=dict(facecolor="black", shrink=0.05, width=1))

plt.xlabel("% of Application Portfolio", fontsize=11)
plt.ylabel("% of Total Technology Spend", fontsize=11)
plt.title("Application Portfolio: Pareto Spend Concentration Curve", fontsize=13)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

---

## 5. Interactive Unit Economics Explorer (Plotly)

We create an interactive Plotly bubble chart showing TCO vs Active Users, sized by Annual Cost per User:

In [4]:
fig = px.scatter(
    app_tco_df,
    x="avg_monthly_users",
    y="annual_tco",
    size="annual_cost_per_user",
    color="lifecycle_status",
    hover_name="application_name",
    hover_data={"application_id": True, "annual_tco": ":,.0f", "avg_monthly_users": ":.0f", "annual_cost_per_user": ":,.2f", "cost_per_transaction": ":.4f"},
    title="Application Unit Economics: Spend vs Scale (Bubble Size = Annual Cost/User)",
    labels={"avg_monthly_users": "Average Monthly Active Users", "annual_tco": "Annual TCO (₹)", "lifecycle_status": "Lifecycle"},
    color_discrete_map={"Strategic": "#2ca02c", "Tolerate": "#ff7f0e", "Migrate": "#1f77b4", "Retire": "#d62728"},
    template="plotly_white"
)
fig.update_layout(yaxis=dict(tickformat="~s"))
fig.show()

---

## 6. Sensitivity Analysis: User Volume Decline Simulation

What happens to unit costs if a corporate divestiture or market shift causes a **30% decline in active user volume** while fixed application costs remain locked?

In [5]:
# Simulate -30% User Volume Shift
sim_df = app_tco_df.copy()
sim_df["simulated_users"] = (sim_df["avg_monthly_users"] * 0.70).round(0)
sim_df["simulated_cost_per_user"] = np.where(
    sim_df["simulated_users"] > 0,
    (sim_df["annual_tco"] / sim_df["simulated_users"]).round(2),
    0.0
)
sim_df["unit_cost_escalation_pct"] = (
    (sim_df["simulated_cost_per_user"] - sim_df["annual_cost_per_user"]) / sim_df["annual_cost_per_user"] * 100.0
).round(1)

print("Sensitivity Simulation: Impact of -30% User Adoption on Unit Cost:")
display(sim_df[["application_id", "application_name", "avg_monthly_users", "simulated_users", "annual_cost_per_user", "simulated_cost_per_user", "unit_cost_escalation_pct"]].head(6))

Sensitivity Simulation: Impact of -30% User Adoption on Unit Cost:


---

## 7. Active Development (v2): Advanced Activity-Based Costing & Unit Economics

In the **v2 Architecture**, we elevate application cost analytics to enterprise FinOps standards:
1. **2-Stage Activity-Based Costing (ABC):** Decomposes resource costs into Software Engineering, Platform Resiliency, Transaction Processing, and Helpdesk Operations.
2. **Contractual Price Escalation & Inflation Sensitivity:** Projects multi-year TCO trajectory under varying macro-inflation scenarios.
3. **FinOps Unit Economics Benchmarking:** Evaluates applications against industry percentiles to identify leaders and optimization candidates.
4. **Econometric TCO Forecasting:** Projects forward-looking monthly TCO with statistical 95% confidence intervals.

In [6]:
from tvi.cost_analytics import (
    calculate_activity_based_costing,
    model_price_escalation_sensitivity,
    benchmark_finops_unit_economics,
    forecast_application_tco,
)

# 1. Activity-Based Costing Decomposition
abc_df = calculate_activity_based_costing()
print(f"Computed 2-Stage ABC Cost Allocations for {len(abc_df)} applications:")
display(abc_df.head(6))

# 2. Price Escalation & Inflation Sensitivity
sensitivity_df = model_price_escalation_sensitivity([0.03, 0.05, 0.08])
print("\n3-Year TCO Trajectory Under Macro-Inflation Scenarios:")
display(sensitivity_df)

# 3. FinOps Unit Economics Benchmarking
finops_df = benchmark_finops_unit_economics()
print("\nFinOps Maturity Benchmark Breakdown:")
print(finops_df["finops_efficiency_tier"].value_counts().to_string())
display(finops_df[["application_id", "application_name", "annual_tco", "annual_cost_per_user", "cost_per_transaction", "finops_efficiency_tier"]].head(6))

# 4. Econometric TCO Forecasting for Scale Anchor APP001
forecast_df = forecast_application_tco("APP001", forward_periods=6)
print(f"\nForward 6-Month TCO Forecast for APP001 (95% Confidence Interval):")
display(forecast_df)

Computed 2-Stage ABC Cost Allocations for 50 applications:

3-Year TCO Trajectory Under Macro-Inflation Scenarios:

FinOps Maturity Benchmark Breakdown:
finops_efficiency_tier
Optimization Candidate           16
Top Quartile (FinOps Leader)     16
Critical Review (Inefficient)    11
Industry Par (Efficient)          7

Forward 6-Month TCO Forecast for APP001 (95% Confidence Interval):


## 7. Limitations & Analytical Guardrails

1. **Fixed vs Variable Cost Realities:** Unit costs (TCO / users) assume simple proportional allocation. In reality, a large portion of software license and infrastructure cost is fixed. A 50% decrease in active users does not automatically reduce total application spend by 50%.
2. **Non-User Applications:** Middleware, batch ETL systems, and security daemons deliver enterprise value without direct interactive end-users; their unit economics must be evaluated via transaction and API volumes.

## 8. Next Step

In **Notebook 05: Capability Cost Intelligence**, we elevate our financial lens from individual applications to **Business Capabilities**, solving the critical challenge of allocating shared IT services without double-counting.